# UC-ASR on Kaggle GPU -- cache-fill batch run

Strategy: a Kaggle **T4** does only the expensive part -- the real LLM scoring -- and writes every
reasoning trace + score into the permanent SQLite cache `cache/llm_responses.sqlite`. You then
**download that cache** and run eval/figures locally in seconds (every LLM call is a cache hit).

Scale = `kaggle_unc`: real Qwen2.5-3B 4-bit, K=50, N=3 @ T=0.7, 150 eval users, 1 seed, 3 uncertainty
ablations, 256 max_new_tokens. Produces a complete publishable uncertainty-method result in ~1-2h GPU.

**Before running:** Settings -> Accelerator -> **GPU T4** ; Settings -> **Internet: On**.
The SQLite cache makes any re-run resume for free, so a session timeout never loses work.


## 1. Clone the repo


In [ ]:
REPO_URL = 'https://github.com/ArunMuthukumarasamy/UC_ASR_Thesis.git'
import os, subprocess
WORK = '/kaggle/working/UC_ASR_Thesis'
if not os.path.isdir(WORK):
    subprocess.run(['git','clone','--depth','1',REPO_URL,WORK], check=True)
os.chdir(WORK)
print('cwd =', os.getcwd()); print(sorted(os.listdir('.')))


### (Optional) resume from a previously downloaded cache
If you attached a Kaggle Dataset containing an earlier `llm_responses.sqlite`, copy it in so this
run resumes instead of re-scoring. Skip if starting fresh.


In [ ]:
# import shutil
# os.makedirs('cache', exist_ok=True)
# shutil.copy('/kaggle/input/<your-cache-slug>/llm_responses.sqlite', 'cache/llm_responses.sqlite')


## 2. Install deps (bitsandbytes works here: Kaggle is Linux+CUDA)


In [ ]:
!pip -q install -r requirements.txt
!pip -q install bitsandbytes


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')


## 3. Sanity-check the kaggle_unc scale


In [ ]:
from src.common import load_config, _on_kaggle
cfg = load_config('config.yaml', 'kaggle_unc')
print('on kaggle :', _on_kaggle())
print('backend   :', cfg['llm']['backend'], '| model:', cfg['llm']['hf_model'])
print('4bit/dev  :', cfg['llm']['load_in_4bit'], '/', cfg['llm']['device'])
print('max_new   :', cfg['llm']['max_new_tokens'], '| N passes:', cfg['uncertainty']['uaro']['n_passes'])
print('users     :', cfg['preprocessing']['n_users_subset'], '| top_k:', cfg['retrievers']['top_k_candidates'])


## 4. Data + retrieve (fast; CPU-bound, minutes)


In [ ]:
!python run_pipeline.py --scale kaggle_unc --stage data


In [ ]:
!python run_pipeline.py --scale kaggle_unc --stage retrieve


## 5. Rerank == FILL THE CACHE (the GPU-heavy part, ~1-2h)
This is the whole reason for the GPU. It writes every real LLM score into `cache/llm_responses.sqlite`.


In [ ]:
!python run_pipeline.py --scale kaggle_unc --stage rerank


## 6. Confirm the cache filled


In [ ]:
import sqlite3
c = sqlite3.connect('cache/llm_responses.sqlite')
tot = c.execute('SELECT COUNT(*) FROM llm_cache').fetchone()[0]
by = c.execute("SELECT model, COUNT(*) FROM llm_cache GROUP BY model ORDER BY 2 DESC LIMIT 5").fetchall()
dflt = c.execute("SELECT COUNT(*) FROM llm_cache WHERE model LIKE '%Qwen%' AND relevance=3.0").fetchone()[0]
print('total cached:', tot); print('by model:', by); print('Qwen fallback-3.0 (lower is better):', dflt)
c.close()


## 7. (Optional) run eval on Kaggle too
Cheap now that the cache is warm -- every call is a cache hit. Or skip and do it locally.


In [ ]:
!python run_pipeline.py --scale kaggle_unc --stage eval


In [ ]:
import pandas as pd, pathlib
p = pathlib.Path('results/thesis_main_table.csv')
print(pathlib.Path('results/thesis_main_table.md').read_text()) if pathlib.Path('results/thesis_main_table.md').exists() else None
display(pd.read_csv(p)) if p.exists() else print('no table yet')


## 8. DOWNLOAD THE CACHE
**Save Version** persists `/kaggle/working`. After it finishes, download **`cache/llm_responses.sqlite`**
(and optionally `data/processed/*.parquet` and `results/`) from the version Output tab. Put the sqlite at
`cache/llm_responses.sqlite` on your laptop, then run locally:
```
python run_pipeline.py --scale kaggle_unc --stage retrieve
python run_pipeline.py --scale kaggle_unc --stage eval
```
Both are instant (all LLM calls are cache hits) and produce the tables, Wilcoxon, and 300-DPI figures.


In [ ]:
!du -sh cache/llm_responses.sqlite data/processed results 2>/dev/null || true
